# CAR-bench Agent Alignment: Supervised Fine-Tuning (SFT) and Direct Preference Optimization (DPO) Pipeline (Local Offline Workstation)

This notebook contains a sequential training pipeline to fine-tune and align an LLM for the CAR-bench voice assistant competition. The pipeline consists of two stages:
1. Supervised Fine-Tuning (SFT): Fine-tunes the base model on successful trajectories to learn general tool-calling and dialogue styles.
2. Direct Preference Optimization (DPO): Uses preference pairs (chosen and rejected responses) to align the model against common policy violations.

## Contents
1. Environment Setup: Install packages and clone repository files.
2. Risk Management: Establish connection keep-alive, persistent storage, and memory recovery safeguards.
3. Synthetic preference generation: Use an OpenAI-compatible API to generate rejected turns by intentionally violating policy rules.
4. Data Preparation: Format the generated preference pairs for SFT and DPO stages.
5. Stage 1 SFT: Load base model weights and execute Supervised Fine-Tuning.
6. Stage 2 DPO: Load SFT model checkpoint and execute Direct Preference Optimization.
7. Visualization: Plot loss, reward accuracy, and reward margins for both SFT and DPO training runs.
8. Model Export: Merge LoRA weights and export to GGUF format.
9. Downloading Results and Checkpoints: Zip and download artifacts.

## CAR-bench Leaderboard & Baselines

Below are the baseline results for frontier and open-source models on the CAR-bench voice assistant tasks. The goal of this alignment pipeline is to fine-tune a smaller, local model (e.g. Qwen2.5-3B-Instruct) to match or exceed these baselines by specializing on the 19 vehicle assistant policies:

| Model | Alignment Method | Avg Pass^3 | Base Pass^3 | Hall. Pass^3 | Disamb. Pass^3 | Source/Platform |
|---|---|---|---|---|---|---|
| Claude Opus 4.6 | RLHF (Frontier) | 0.58 | 0.80 | 0.48 | 0.46 | Anthropic (Official) |
| GPT-5 | RLHF (Frontier) | 0.54 | 0.66 | 0.60 | 0.36 | OpenAI (Official) |
| Gemini 2.5 Pro | RLHF (Frontier) | 0.38 | 0.53 | 0.34 | 0.28 | Google (Official) |
| **Qwen2.5-3B-Instruct (Fine-Tuned)** | **SFT + DPO (This Run)** | **TBD** | **TBD** | **TBD** | **TBD** | **Local / Cloud Run** |
| Qwen3-32B | Base SFT/RLHF | 0.31 | 0.45 | 0.27 | 0.22 | Alibaba (Official) |
| xLAM-2-32B | SFT / DPO | 0.16 | 0.26 | 0.11 | 0.12 | Salesforce (Official) |

## 1. Environment Setup

In [ ]:
# Running in local offline workstation environment
IN_COLAB = False
print("Running on local workstation. Repositories and paths are assumed local.")

In [ ]:
# Ensure you have installed these packages in your local environment:
# pip install transformers peft trl datasets accelerate bitsandbytes huggingface_hub protobuf matplotlib seaborn
# pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
print("Please verify packages are pre-installed in your virtual environment.")

In [ ]:
import os
import sys
import json
import torch
import random

# Establish seeds for training reproducibility
random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

## 2. Risk Management & Persistent Storage

To prevent training failures due to timeouts, RAM overflows, or cloud VM disconnects, we establish the following safeguards:
1. Keep-Alive Script: Javascript to prevent notebook inactivity disconnects.
2. Google Drive Mounting: Saves checkpoints directly to persistent storage (Google Drive or Kaggle outputs) rather than transient VM storage.
3. Memory Cleanup: Garbage collection and GPU cache purge to avoid Out-Of-Memory (OOM) errors.

### Keep-Alive Script
Paste the following JavaScript code into the browser developer console (F12) to simulate clicks and prevent timeouts during long training sessions:

```javascript
function KeepAlive() {
    console.log("Simulating interaction to keep session active...");
    var connectButton = document.querySelector("colab-connect-button");
    if (connectButton) connectButton.click();
}
setInterval(KeepAlive, 60000);
```

In [ ]:
# Local output directory configured for local PC
PERSISTENT_DIR = "./outputs"
os.makedirs(PERSISTENT_DIR, exist_ok=True)
print(f"Target checkpoints directory configured locally: {PERSISTENT_DIR}")

In [ ]:
import gc

def clear_gpu_memory():
    """Purges system garbage collection and releases cached PyTorch CUDA memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("Released GPU cache memory.")

clear_gpu_memory()

## 3. LLM Rejected Preference Pair Generator

If you do not have a preference dataset containing chosen and rejected samples, this section uses an OpenAI-compatible API to rewrite successful assistant responses into policy-violating rejected turns.

In [ ]:
from openai import OpenAI

# Set API keys and Base URL (can be pointed to OpenAI, Gemini, or a local vLLM endpoint)
API_KEY = os.environ.get("OPENAI_API_KEY", "your-api-key")
BASE_URL = os.environ.get("OPENAI_API_BASE", "https://api.openai.com/v1")

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

# Policies to violate
VIOLATION_RULES = [
    "Call a tool without getting a prior confirmation from the user (violates user confirmation rule).",
    "Call a planning_tool directly (violates forbidden tool rule).",
    "Output reasoning thoughts in plaintext rather than JSON (violates output format rule).",
    "Return incorrect coordinates or search fields (violates accuracy rule)."
]

def generate_rejected_response(chosen_content, rule):
    system_instruction = (
        "You are a database alignment assistant. Your task is to generate negative training examples "
        "(rejected responses) for a preference optimization dataset."
    )
    user_prompt = f"""Take the following successful assistant response and rewrite it so that it violates this rule:
Rule: {rule}

Successful Assistant Response:
{chosen_content}

Return ONLY the rewritten response. Do not include explanations, preambles, or conversational formatting. Keep the tool calls/JSON structure where possible, but inject the rule violation."""
    
    try:
        chat_completion = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.7,
            max_tokens=800
        )
        return chat_completion.choices[0].message.content.strip()
    except Exception as e:
        print(f"API Generation failed: {e}")
        return None

In [ ]:
sft_dataset_path = "data/ft_dataset.jsonl"
output_dpo_path = "data/dpo_generated_dataset.jsonl"

# If running in Colab, request upload of sft_dataset if missing
if not os.path.exists(sft_dataset_path):
    try:
        print(f"Please upload your SFT dataset file '{sft_dataset_path}':")
            if fn.endswith(".jsonl"):
                sft_dataset_path = fn
    except Exception as e:
        print("Colab interactive upload not available.")

def construct_dpo_dataset_via_api():
    if not os.path.exists(sft_dataset_path):
        print("SFT dataset not found. Skipping preference pair generation.")
        return
        
    print(f"Reading SFT dataset from {sft_dataset_path}...")
    with open(sft_dataset_path, "r", encoding="utf-8") as f:
        sft_lines = [json.loads(line) for line in f]
        
    dpo_samples = []
    print(f"Generating DPO preference pairs for {len(sft_lines)} dialogues...")
    for line in sft_lines:
        messages = line.get("messages", [])
        if len(messages) < 2:
            continue
            
        # Extract last assistant response as the chosen response
        last_idx = -1
        for idx in range(len(messages) - 1, -1, -1):
            if messages[idx].get("role") == "assistant":
                last_idx = idx
                break
                
        if last_idx == -1:
            continue
            
        prompt_history = messages[:last_idx]
        chosen_turn = messages[last_idx]
        chosen_content = chosen_turn.get("content") or ""
        
        # Choose a random rule to violate
        rule = random.choice(VIOLATION_RULES)
        rejected_content = generate_rejected_response(chosen_content, rule)
        
        if rejected_content:
            rejected_turn = chosen_turn.copy()
            rejected_turn["content"] = rejected_content
            
            dpo_samples.append({
                "prompt": prompt_history,
                "chosen": [chosen_turn],
                "rejected": [rejected_turn]
            })
            time.sleep(0.5) # Sleep to respect API rate limits
            
    with open(output_dpo_path, "w", encoding="utf-8") as out_f:
        for sample in dpo_samples:
            out_f.write(json.dumps(sample) + "\n")
    print(f"Saved {len(dpo_samples)} DPO samples to {output_dpo_path}")

# Uncomment the following line to run DPO generation via LLM API:
# construct_dpo_dataset_via_api()

## 4. Data Preparation

We load training datasets for both the SFT and DPO training phases.

In [ ]:
from datasets import Dataset, load_dataset

dpo_dataset_path = "data/dpo_dataset.jsonl"
if os.path.exists(output_dpo_path):
    dpo_dataset_path = output_dpo_path

sft_dataset = None
dpo_dataset = None

# Load datasets
if os.path.exists(sft_dataset_path):
    print(f"Loading SFT dataset from {sft_dataset_path}...")
    sft_dataset = load_dataset("json", data_files=sft_dataset_path, split="train")
if os.path.exists(dpo_dataset_path):
    print(f"Loading DPO dataset from {dpo_dataset_path}...")
    dpo_dataset = load_dataset("json", data_files=dpo_dataset_path, split="train")

print(f"SFT Samples: {len(sft_dataset) if sft_dataset else 0}")
print(f"DPO Samples: {len(dpo_dataset) if dpo_dataset else 0}")

## 5. Stage 1: Supervised Fine-Tuning (SFT)

We fine-tune the base instructions using standard chat format on successful trajectories.

In [ ]:
# RESTRICTED ORDER OF IMPORTS - Unsloth must be imported BEFORE transformers/peft/trl
try:
    import unsloth
except ImportError:
    pass

from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MAX_SEQ_LENGTH = 2048

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_sft_data(example):
    formatted_text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": formatted_text}

if sft_dataset:
    sft_dataset = sft_dataset.map(format_sft_data)


In [ ]:
# Pre-loading clean memory check
clear_gpu_memory()

from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments

print("Loading base model via Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    trust_remote_code=True
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

# Setup output directory for Stage 1
sft_checkpoint_dir = os.path.join(PERSISTENT_DIR, "sft")
os.makedirs(sft_checkpoint_dir, exist_ok=True)

# Check for existing SFT checkpoints to auto-resume
sft_resume_checkpoint = None
checkpoints = [os.path.join(sft_checkpoint_dir, d) for d in os.listdir(sft_checkpoint_dir) if d.startswith("checkpoint-")]
if checkpoints:
    checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
    sft_resume_checkpoint = checkpoints[-1]
    print(f"Found SFT checkpoint: {sft_resume_checkpoint}. Resuming SFT training...")

sft_training_args = TrainingArguments(
    output_dir=sft_checkpoint_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    logging_steps=5,
    num_train_epochs=2,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_ratio=0.03,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="paged_adamw_8bit",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    average_tokens_across_devices=False
)

sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=sft_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=sft_training_args
)

if sft_dataset:
    print("Starting Stage 1: SFT Training...")
    try:
        sft_stats = sft_trainer.train(resume_from_checkpoint=sft_resume_checkpoint)
        print("SFT training completed. Saving adapter weights...")
        model.save_pretrained(os.path.join(PERSISTENT_DIR, "sft_adapter"))
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print("CUDA Out Of Memory in SFT. Reduce max_seq_length or batch size.")
        raise e
else:
    print("SFT dataset missing. Skipping Stage 1 training.")


## 6. Stage 2: Direct Preference Optimization (DPO)

We load the adapter weights from the SFT phase and perform Direct Preference Optimization to align the model's behavior against policy rules.

In [ ]:
def format_dpo_data(examples):
    formatted_prompts = []
    formatted_chosen = []
    formatted_rejected = []
    
    for prompt, chosen, rejected in zip(examples["prompt"], examples["chosen"], examples["rejected"]):
        p_text = tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
        c_text = tokenizer.apply_chat_template(chosen, tokenize=False, add_generation_prompt=False)
        r_text = tokenizer.apply_chat_template(rejected, tokenize=False, add_generation_prompt=False)
        
        if c_text.startswith(p_text):
            c_text = c_text[len(p_text):]
        if r_text.startswith(p_text):
            r_text = r_text[len(p_text):]
            
        formatted_prompts.append(p_text)
        formatted_chosen.append(c_text)
        formatted_rejected.append(r_text)
        
    return {
        "prompt": formatted_prompts,
        "chosen": formatted_chosen,
        "rejected": formatted_rejected
    }

if dpo_dataset:
    formatted_dpo_dataset = dpo_dataset.map(format_dpo_data, batched=True)

In [ ]:
# Pre-loading clean memory check
clear_gpu_memory()

from trl import DPOTrainer, DPOConfig
from unsloth import PatchDPOTrainer

# Patch DPOTrainer for memory optimization
PatchDPOTrainer()

# Setup output directory for Stage 2
dpo_checkpoint_dir = os.path.join(PERSISTENT_DIR, "dpo")
os.makedirs(dpo_checkpoint_dir, exist_ok=True)

# Check for existing DPO checkpoints to auto-resume
dpo_resume_checkpoint = None
checkpoints = [os.path.join(dpo_checkpoint_dir, d) for d in os.listdir(dpo_checkpoint_dir) if d.startswith("checkpoint-")]
if checkpoints:
    checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
    dpo_resume_checkpoint = checkpoints[-1]
    print(f'Found DPO checkpoint: {dpo_resume_checkpoint}. Resuming DPO training...')

dpo_config = DPOConfig(
    output_dir=dpo_checkpoint_dir,
    learning_rate=5e-6,
    beta=0.1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="paged_adamw_8bit",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=5,
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=MAX_SEQ_LENGTH // 2,
    report_to="none",
    average_tokens_across_devices=False
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,           # Not required when using patched Unsloth DPOTrainer
    args=dpo_config,
    train_dataset=formatted_dpo_dataset,
    tokenizer=tokenizer
)

if dpo_dataset:
    print("Starting Stage 2: DPO Preferences Alignment...")
    try:
        dpo_stats = dpo_trainer.train(resume_from_checkpoint=dpo_resume_checkpoint)
        print("DPO training completed successfully!")
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print("CUDA Out Of Memory in DPO. Reduce max_prompt_length or batch size.")
        raise e
else:
    print("DPO dataset missing. Skipping Stage 2 preference alignment.")


## 7. Training Metrics Visualization

We plot loss and accuracy curves for both SFT and DPO runs using matplotlib.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 5))

# 1. Plot SFT Training Loss
if sft_dataset and hasattr(sft_trainer, "state") and sft_trainer.state.log_history:
    sft_history = sft_trainer.state.log_history
    sft_steps = [log["step"] for log in sft_history if "step" in log and "loss" in log]
    sft_losses = [log["loss"] for log in sft_history if "step" in log and "loss" in log]
    
    plt.subplot(1, 3, 1)
    plt.plot(sft_steps, sft_losses, label="SFT Loss", color="blue")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title("Stage 1 SFT Loss")
    plt.grid(True)

# 2. Plot DPO Training Loss
if dpo_dataset and hasattr(dpo_trainer, "state") and dpo_trainer.state.log_history:
    dpo_history = dpo_trainer.state.log_history
    dpo_steps = [log["step"] for log in dpo_history if "step" in log and "loss" in log]
    dpo_losses = [log["loss"] for log in dpo_history if "step" in log and "loss" in log]
    
    plt.subplot(1, 3, 2)
    plt.plot(dpo_steps, dpo_losses, label="DPO Loss", color="red")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title("Stage 2 DPO Loss")
    plt.grid(True)
    
# 3. Plot DPO Margins and Accuracy
if dpo_dataset and hasattr(dpo_trainer, "state") and dpo_trainer.state.log_history:
    dpo_history = dpo_trainer.state.log_history
    dpo_acc_steps = [log["step"] for log in dpo_history if "step" in log and "rewards/accuracies" in log]
    dpo_accs = [log["rewards/accuracies"] for log in dpo_history if "step" in log and "rewards/accuracies" in log]
    
    plt.subplot(1, 3, 3)
    plt.plot(dpo_acc_steps, dpo_accs, label="Reward Accuracy", color="green")
    plt.xlabel("Step")
    plt.ylabel("Accuracy")
    plt.title("Stage 2 Reward Accuracy")
    plt.grid(True)

plt.tight_layout()
plt.show()

## 8. Saving and Exporting Model Weights

In [ ]:
ADAPTER_PATH = os.path.join(PERSISTENT_DIR, "lora_adapter_dpo")
MERGED_PATH = os.path.join(PERSISTENT_DIR, "merged_model_dpo")

# Save LoRA adapter weights
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"LoRA adapter weights saved to {ADAPTER_PATH}")

# Save merged model (16-bit precision)
print("Saving merged 16-bit model weights...")
model.save_pretrained_merged(MERGED_PATH, tokenizer, save_method="merged_16bit")
print(f"Merged model saved to {MERGED_PATH}")

In [ ]:
GGUF_PATH = os.path.join(PERSISTENT_DIR, "merged_q4_k_m_dpo")
print("Exporting model to Q4_K_M GGUF format...")
try:
    model.save_pretrained_merged(GGUF_PATH, tokenizer, save_method="quantized_gguf", quantization_method="q4_k_m")
    print(f"GGUF model exported to {GGUF_PATH}-unsloth.Q4_K_M.gguf")
except Exception as e:
    print(f"Export to GGUF failed: {e}")
    
# Provide download link in Google Colab environment


## 9. Downloading Results and Checkpoints

This section provides helper functions to compress (zip) and download model checkpoints, adapter files, or full training outputs. This is especially useful in Google Colab (triggers browser download) and Kaggle (saves zip file to outputs for easy download via sidebar).

In [ ]:
import shutil
import os

def zip_and_download(folder_path, output_filename):
    """Zips a folder and triggers download in Colab, or creates a downloadable file in Kaggle/local."""
    if not os.path.exists(folder_path):
        print(f"Folder '{folder_path}' does not exist. Skipping compression.")
        return
        
    zip_path = f"{output_filename}.zip"
    print(f"Zipping '{folder_path}' into '{zip_path}'...")
    
    # Zip the folder
    shutil.make_archive(output_filename, 'zip', folder_path)
    file_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"Created: {zip_path} ({file_size_mb:.2f} MB)")
    
# Choose what you want to download:
# Option 1: Download LoRA adapter only (recommended, small size)
zip_and_download(ADAPTER_PATH, "lora_adapter_dpo_results")

# Option 2: Download full training outputs/checkpoints folder (large size)
# zip_and_download(PERSISTENT_DIR, "all_training_checkpoints_dpo")